# UAE-SL Dataset Preprocessing Pipeline

Uses **MediaPipe Tasks API** (`HolisticLandmarker`) — not the legacy `mediapipe.solutions` API.

Pipeline stages:
1. **Landmark Extraction** — `HolisticLandmarker` → 75 keypoints/frame
2. **Normalization** — shoulder-width normalization + 60-frame padding
3. **Augmentation** — spatial + temporal (target: 30 samples/class)
4. **Dataset Building** — stratified train / val / test splits

In [9]:
import os
import json
import warnings
import numpy as np
import cv2
import pandas as pd
import mediapipe as mp
from mediapipe.tasks import python as mp_python
from mediapipe.tasks.python import vision as mp_vision
from pathlib import Path
from sklearn.model_selection import train_test_split
from scipy.interpolate import interp1d
from tqdm import tqdm
import tensorflow as tf
import absl.logging

warnings.filterwarnings('ignore')
np.random.seed(42)

os.environ['TF_CPP_MIN_LOG_LEVEL'] = '3'
os.environ['GLOG_minloglevel'] = '3'
os.environ['GRPC_VERBOSITY_LEVEL'] = 'ERROR'


absl.logging.set_verbosity(absl.logging.ERROR)

In [10]:
print(tf.config.list_physical_devices('GPU'))

[PhysicalDevice(name='/physical_device:GPU:0', device_type='GPU')]


## Configuration

In [11]:
# ── Paths ─────────────────────────────────────────────────────────────────────
PROJECT_ROOT   = Path('..').resolve()

# Raw videos: UAE-dataset/{category}/{VideoName.mp4}
RAW_VIDEO_DIR  = PROJECT_ROOT / 'UAE-dataset'

# MediaPipe Tasks model
HOLISTIC_MODEL = PROJECT_ROOT / 'models' / 'holistic_landmarker.task'

# Outputs (mirrors CLAUDE.md dataset/ structure)
LANDMARKS_DIR  = PROJECT_ROOT / 'UAE-dataset' / 'processed' / 'landmarks'       # raw extracted .npy
NORM_DIR       = PROJECT_ROOT / 'UAE-dataset' / 'processed' / 'landmarks_norm'  # normalised .npy
AUGMENTED_DIR  = PROJECT_ROOT / 'UAE-dataset' / 'processed' / 'augmented'       # augmented .npy
SPLITS_DIR     = PROJECT_ROOT / 'UAE-dataset' / 'processed' / 'splits'          # CSVs + class map

for d in [LANDMARKS_DIR, NORM_DIR, AUGMENTED_DIR, SPLITS_DIR]:
    d.mkdir(parents=True, exist_ok=True)

assert HOLISTIC_MODEL.exists(), f'Model not found: {HOLISTIC_MODEL}'
assert RAW_VIDEO_DIR.exists(),  f'UAE-dataset not found: {RAW_VIDEO_DIR}'

# ── Sequence settings ─────────────────────────────────────────────────────────
SEQ_LEN        = 60    # fixed frames output (~2 s at 30 fps)
N_KEYPOINTS    = 75    # 33 pose + 21 left hand + 21 right hand
N_DIMS         = 2     # (x, y) — set to 3 to include z

# ── Augmentation / splits ─────────────────────────────────────────────────────
AUGMENT_N      = 29    # copies per class  →  30 total (1 original + 29 augmented)
VAL_SPLIT      = 0.15
TEST_SPLIT     = 0.10
SEED           = 42

print(f'Model  : {HOLISTIC_MODEL}')
print(f'Videos : {RAW_VIDEO_DIR}')

Model  : /home/livex/projects/Sign2Chat/models/holistic_landmarker.task
Videos : /home/livex/projects/Sign2Chat/UAE-dataset


---
## Stage 1 — Landmark Extraction (MediaPipe Tasks API)

Uses `HolisticLandmarker` in **VIDEO** running mode for per-frame tracking.

In [12]:
def build_holistic_landmarker() -> mp_vision.HolisticLandmarker:
    """Create a HolisticLandmarker in VIDEO mode (reusable across frames)."""
    options = mp_vision.HolisticLandmarkerOptions(
        base_options=mp_python.BaseOptions(
            model_asset_path=str(HOLISTIC_MODEL)
        ),
        running_mode=mp_vision.RunningMode.VIDEO,
        min_pose_detection_confidence=0.5,
        min_pose_suppression_threshold=0.5,
        min_pose_landmarks_confidence=0.5,
        min_face_detection_confidence=0.5,
        min_face_suppression_threshold=0.5,
        min_face_landmarks_confidence=0.5,
        min_hand_landmarks_confidence=0.5,
        output_face_blendshapes=False,
    )
    return mp_vision.HolisticLandmarker.create_from_options(options)


def _lm_list_to_array(landmark_list, n: int) -> np.ndarray:
    """Convert a Tasks API landmark list to (n, N_DIMS) float32 array."""
    if not landmark_list:
        return np.zeros((n, N_DIMS), dtype=np.float32)
    pts = []
    for lm in landmark_list:
        row = [lm.x, lm.y] if N_DIMS == 2 else [lm.x, lm.y, lm.z]
        pts.append(row)
    return np.array(pts[:n], dtype=np.float32)


def parse_holistic_result(result) -> tuple[np.ndarray, bool]:
    """
    Convert a HolisticLandmarkerResult to (75, N_DIMS) array.

    Returns
    -------
    keypoints : ndarray  shape (75, N_DIMS)
    hand_detected : bool  — True when at least one hand was found
    """
    # Pose: result.pose_landmarks is List[List[NormalizedLandmark]]
    pose_lms  = result.pose_landmarks       if result.pose_landmarks       else []
    lhand_lms = result.left_hand_landmarks   if result.left_hand_landmarks   else []
    rhand_lms = result.right_hand_landmarks  if result.right_hand_landmarks  else []

    pose  = _lm_list_to_array(pose_lms,  33)
    lhand = _lm_list_to_array(lhand_lms, 21)
    rhand = _lm_list_to_array(rhand_lms, 21)

    hand_detected = bool(lhand_lms or rhand_lms)
    return np.concatenate([pose, lhand, rhand], axis=0), hand_detected


def extract_video_landmarks(video_path: Path) -> np.ndarray | None:
    """
    Extract raw landmark sequence from a single video file.

    Returns ndarray of shape (T, 75, N_DIMS), or None on failure.
    Frames with no hand detected are zero-filled and later interpolated.
    """
    cap = cv2.VideoCapture(str(video_path))
    if not cap.isOpened():
        return None

    fps = cap.get(cv2.CAP_PROP_FPS) or 30.0
    frames, no_hand_mask = [], []
    frame_idx = 0

    with build_holistic_landmarker() as landmarker:
        while True:
            ret, bgr = cap.read()
            if not ret:
                break

            rgb = cv2.cvtColor(bgr, cv2.COLOR_BGR2RGB)
            mp_image = mp.Image(image_format=mp.ImageFormat.SRGB, data=rgb)

            # Timestamp must be strictly monotonically increasing (ms)
            timestamp_ms = int(frame_idx * 1000 / fps)
            result = landmarker.detect_for_video(mp_image, timestamp_ms)

            kp, hand_ok = parse_holistic_result(result)
            frames.append(kp)
            no_hand_mask.append(not hand_ok)
            frame_idx += 1

    cap.release()

    if not frames:
        return None

    seq  = np.stack(frames, axis=0).astype(np.float32)   # (T, 75, N_DIMS)
    mask = np.array(no_hand_mask, dtype=bool)             # True = no hand

    # Interpolate hand joints (indices 33–74) for no-hand frames
    good_idx = np.where(~mask)[0]
    if len(good_idx) > 1 and mask.any():
        bad_idx = np.where(mask)[0]
        for j in range(33, 75):
            for d in range(N_DIMS):
                interp = interp1d(
                    good_idx, seq[good_idx, j, d],
                    kind='linear',
                    bounds_error=False,
                    fill_value=(seq[good_idx[0], j, d], seq[good_idx[-1], j, d])
                )
                seq[bad_idx, j, d] = interp(bad_idx)

    return seq

In [31]:
# Collect all videos: UAE-dataset/{category}/{VideoName.mp4}
video_exts = {'.mp4', '.avi', '.mov', '.mkv', '.webm'}

video_files = [
    f
    for category in sorted(RAW_VIDEO_DIR.iterdir()) if category.is_dir()
    for f in sorted(category.iterdir()) if f.suffix.lower() in video_exts
]

print(f'Found {len(video_files)} videos across {len(list(RAW_VIDEO_DIR.iterdir()))} categories')
for cat in sorted(RAW_VIDEO_DIR.iterdir()):
    n = sum(1 for f in cat.iterdir() if f.suffix.lower() in video_exts)
    print(f'  {cat.name}: {n} videos')

Found 1246 videos across 22 categories
  Alphabet: 31 videos
  Animals: 64 videos
  Attributes & Situations: 77 videos
  Clothing & Toiletries: 89 videos
  Colors: 19 videos
  Common Verbs: 147 videos
  Cuisines: 29 videos
  Directions & Locations: 24 videos
  Education: 61 videos
  Enviroment: 68 videos
  Family: 24 videos
  Health: 88 videos
  Household: 101 videos
  Landmarks & Locations: 27 videos
  Measurements: 33 videos
  Ministries: 42 videos
  Numbers: 116 videos
  Official Documents: 36 videos
  Plants: 24 videos
  Professions: 100 videos
  Sports: 46 videos
  processed: 0 videos


In [35]:
failed = []

for video_path in tqdm(video_files, desc='Extracting landmarks'):
    class_name = video_path.stem          # e.g. 'Alphabets_01_Alif'
    out_path   = LANDMARKS_DIR / f'{class_name}.npy'

    if out_path.exists():
        continue

    seq = extract_video_landmarks(video_path)
    if seq is None:
        failed.append(video_path.name)
        continue

    np.save(out_path, seq)

extracted = sorted(LANDMARKS_DIR.glob('*.npy'))
print(f'\nExtracted : {len(extracted)}  |  Failed : {len(failed)}')
if failed:
    print('Failed:', failed)

Extracting landmarks: 100%|██████████| 1246/1246 [00:00<00:00, 24662.95it/s]



Extracted : 1245  |  Failed : 1
Failed: ['Health_83_Avoid.mp4']


---
## Stage 2 — Normalization & Padding

In [36]:
# MediaPipe Pose joint indices (0-based, within the 75-joint array)
LEFT_SHOULDER  = 11
RIGHT_SHOULDER = 12


def normalize_shoulder_width(seq: np.ndarray) -> np.ndarray:
    """
    Translate origin to shoulder midpoint; scale so shoulder width == 1.
    seq: (T, 75, N_DIMS)  →  (T, 75, N_DIMS)
    """
    seq = seq.copy()
    ls = seq[:, LEFT_SHOULDER,  :2]   # (T, 2)
    rs = seq[:, RIGHT_SHOULDER, :2]   # (T, 2)

    widths = np.linalg.norm(ls - rs, axis=1)         # (T,)
    valid  = widths[widths > 1e-6]
    if len(valid) == 0:
        return seq                                    # no pose detected; skip

    mean_width   = float(valid.mean())
    mean_midpoint = ((ls + rs) / 2).mean(axis=0)     # (2,)

    seq[:, :, 0] = (seq[:, :, 0] - mean_midpoint[0]) / mean_width
    seq[:, :, 1] = (seq[:, :, 1] - mean_midpoint[1]) / mean_width
    if N_DIMS == 3:
        seq[:, :, 2] = seq[:, :, 2] / mean_width
    return seq


def pad_or_truncate(seq: np.ndarray, target: int = SEQ_LEN) -> np.ndarray:
    """
    Resize sequence to exactly `target` frames.
    Truncate: evenly-spaced sampling.  Pad: repeat last frame.
    """
    T = seq.shape[0]
    if T == target:
        return seq
    if T > target:
        idx = np.linspace(0, T - 1, target, dtype=int)
        return seq[idx]
    pad = np.repeat(seq[-1:], target - T, axis=0)
    return np.concatenate([seq, pad], axis=0)


def preprocess(seq: np.ndarray) -> np.ndarray:
    seq = normalize_shoulder_width(seq)
    seq = pad_or_truncate(seq)
    return seq.astype(np.float32)   # (60, 75, N_DIMS)

In [37]:
raw_files = sorted(LANDMARKS_DIR.glob('*.npy'))

for fpath in tqdm(raw_files, desc='Normalizing'):
    out = NORM_DIR / fpath.name
    if out.exists():
        continue
    seq = preprocess(np.load(fpath))
    np.save(out, seq)

sample = np.load(next(NORM_DIR.glob('*.npy')))
print(f'Shape : {sample.shape}  expected ({SEQ_LEN}, {N_KEYPOINTS}, {N_DIMS})')
assert sample.shape == (SEQ_LEN, N_KEYPOINTS, N_DIMS), 'Shape mismatch!'
print('Normalization OK')

Normalizing:   0%|          | 0/1245 [00:00<?, ?it/s]

Normalizing: 100%|██████████| 1245/1245 [00:00<00:00, 1329.80it/s]

Shape : (60, 75, 2)  expected (60, 75, 2)
Normalization OK


---
## Stage 3 — Augmentation

Each transform is a standalone function on `(60, 75, N_DIMS)` arrays.

In [38]:
# ── Spatial transforms ────────────────────────────────────────────────────────

def aug_scale(seq: np.ndarray, low: float = 0.85, high: float = 1.15) -> np.ndarray:
    """Uniform random scaling ±15%."""
    return seq * np.random.uniform(low, high)


def aug_mirror(seq: np.ndarray) -> np.ndarray:
    """
    Left/right flip: negate x, swap left↔right hand blocks,
    and swap mirrored pose joint pairs (MediaPipe convention).
    """
    seq = seq.copy()
    seq[:, :, 0] = -seq[:, :, 0]

    # Hand blocks: lhand = joints 33-53, rhand = joints 54-74
    lhand = seq[:, 33:54, :].copy()
    seq[:, 33:54, :] = seq[:, 54:75, :]
    seq[:, 54:75, :] = lhand

    # Symmetric pose pairs (MediaPipe 33-joint index)
    pairs = [
        (1,4),(2,5),(3,6),(7,8),(9,10),(11,12),(13,14),(15,16),
        (17,18),(19,20),(21,22),(23,24),(25,26),(27,28),(29,30),(31,32)
    ]
    for l, r in pairs:
        seq[:, l, :], seq[:, r, :] = seq[:, r, :].copy(), seq[:, l, :].copy()
    return seq


def aug_rotate(seq: np.ndarray, max_deg: float = 10.0) -> np.ndarray:
    """2-D in-plane rotation on x/y (±10°)."""
    angle = np.random.uniform(-max_deg, max_deg) * np.pi / 180.0
    c, s  = np.cos(angle), np.sin(angle)
    R = np.array([[c, -s], [s, c]], dtype=np.float32)
    seq = seq.copy()
    seq[:, :, :2] = seq[:, :, :2] @ R.T
    return seq


def aug_translate(seq: np.ndarray, max_shift: float = 0.10) -> np.ndarray:
    """Random x/y translation (simulate camera position shift)."""
    dx = np.random.uniform(-max_shift, max_shift)
    dy = np.random.uniform(-max_shift, max_shift)
    seq = seq.copy()
    seq[:, :, 0] += dx
    seq[:, :, 1] += dy
    return seq


# ── Temporal transforms ───────────────────────────────────────────────────────

def aug_speed(seq: np.ndarray, low: float = 0.80, high: float = 1.20) -> np.ndarray:
    """Speed perturbation ±20%: resample then pad/truncate to SEQ_LEN."""
    T      = seq.shape[0]
    factor = np.random.uniform(low, high)
    new_T  = max(2, int(round(T * factor)))
    old_t  = np.arange(T)
    new_t  = np.linspace(0, T - 1, new_T)
    out    = np.zeros((new_T, N_KEYPOINTS, N_DIMS), dtype=np.float32)
    for j in range(N_KEYPOINTS):
        for d in range(N_DIMS):
            out[:, j, d] = interp1d(old_t, seq[:, j, d], kind='linear')(new_t)
    return pad_or_truncate(out)


def aug_drop_frames(seq: np.ndarray, max_drop_rate: float = 0.10) -> np.ndarray:
    """Zero out a random fraction of frames (simulate detection gaps)."""
    seq    = seq.copy()
    n_drop = max(1, int(seq.shape[0] * np.random.uniform(0, max_drop_rate)))
    idx    = np.random.choice(seq.shape[0], n_drop, replace=False)
    seq[idx] = 0.0
    return seq


def aug_noise(seq: np.ndarray, sigma: float = 0.01) -> np.ndarray:
    """Gaussian noise on all landmark coordinates."""
    return seq + np.random.normal(0, sigma, seq.shape).astype(np.float32)


# ── Combined sampler ──────────────────────────────────────────────────────────

def random_augment(seq: np.ndarray) -> np.ndarray:
    """
    Randomly apply 1–2 spatial and 1–2 temporal transforms.
    Mirror is applied independently with p=0.5.
    """
    spatial  = [aug_scale, aug_rotate, aug_translate]
    temporal = [aug_speed, aug_drop_frames, aug_noise]

    for fn in np.random.choice(spatial,  size=np.random.randint(1, 3), replace=False):
        seq = fn(seq)
    if np.random.random() < 0.5:
        seq = aug_mirror(seq)
    for fn in np.random.choice(temporal, size=np.random.randint(1, 3), replace=False):
        seq = fn(seq)

    return seq.astype(np.float32)

In [39]:
norm_files = sorted(NORM_DIR.glob('*.npy'))
print(f'Augmenting {len(norm_files)} classes  ×  {AUGMENT_N} copies  →  {len(norm_files) * (AUGMENT_N + 1)} total samples')

for fpath in tqdm(norm_files, desc='Augmenting'):
    class_name = fpath.stem
    seq = np.load(fpath)

    # aug_00 = original
    orig_out = AUGMENTED_DIR / f'{class_name}_aug_00.npy'
    if not orig_out.exists():
        np.save(orig_out, seq)

    for i in range(1, AUGMENT_N + 1):
        out = AUGMENTED_DIR / f'{class_name}_aug_{i:02d}.npy'
        if not out.exists():
            np.save(out, random_augment(seq))

print(f'Total augmented files: {len(list(AUGMENTED_DIR.glob("*.npy")))}')

Augmenting 1245 classes  ×  29 copies  →  37350 total samples


Augmenting:   0%|          | 0/1245 [00:00<?, ?it/s]

Augmenting: 100%|██████████| 1245/1245 [01:50<00:00, 11.24it/s]


Total augmented files: 37350


---
## Stage 4 — Dataset Splits

In [40]:
# Build metadata table
records = []
for f in sorted(AUGMENTED_DIR.glob('*.npy')):
    stem  = f.stem                      # e.g. 'Alphabets_01_Alif_aug_03'
    parts = stem.rsplit('_aug_', 1)
    records.append({
        'file'      : str(f),
        'class'     : parts[0],
        'aug_id'    : int(parts[1]) if len(parts) == 2 else 0,
    })

df = pd.DataFrame(records)
classes   = sorted(df['class'].unique())
class2idx = {c: i for i, c in enumerate(classes)}
df['label'] = df['class'].map(class2idx)

print(f'Classes : {len(classes)}  |  Total samples : {len(df)}')
print(df.groupby('class').size().describe().to_string())

Classes : 1245  |  Total samples : 37350
count    1245.0
mean       30.0
std         0.0
min        30.0
25%        30.0
50%        30.0
75%        30.0
max        30.0


In [41]:
# Split strategy:
#   - aug_00 (original) always goes to train
#   - aug_01…aug_N are stratified-split into train / val / test

originals = df[df['aug_id'] == 0].copy()
augmented = df[df['aug_id'] != 0].copy()

aug_trainval, aug_test = train_test_split(
    augmented, test_size=TEST_SPLIT,
    stratify=augmented['label'], random_state=SEED
)
relative_val = VAL_SPLIT / (1.0 - TEST_SPLIT)
aug_train, aug_val = train_test_split(
    aug_trainval, test_size=relative_val,
    stratify=aug_trainval['label'], random_state=SEED
)

train_df = pd.concat([originals, aug_train]).sample(frac=1, random_state=SEED).reset_index(drop=True)
val_df   = aug_val.reset_index(drop=True)
test_df  = aug_test.reset_index(drop=True)

train_df.to_csv(SPLITS_DIR / 'train.csv', index=False)
val_df.to_csv(  SPLITS_DIR / 'val.csv',   index=False)
test_df.to_csv( SPLITS_DIR / 'test.csv',  index=False)

with open(SPLITS_DIR / 'class_map.json', 'w', encoding='utf-8') as f:
    json.dump(class2idx, f, indent=2, ensure_ascii=False)

print(f'Train : {len(train_df)}  |  Val : {len(val_df)}  |  Test : {len(test_df)}')
print(f'Class map saved → {SPLITS_DIR / "class_map.json"}')

Train : 28323  |  Val : 5416  |  Test : 3611
Class map saved → /home/livex/projects/Sign2Chat/UAE-dataset/processed/splits/class_map.json


---
## Verification

In [42]:
print('=== Pipeline Summary ===')
print(f'  Classes         : {len(classes)}')
print(f'  Train samples   : {len(train_df)}')
print(f'  Val   samples   : {len(val_df)}')
print(f'  Test  samples   : {len(test_df)}')
print(f'  Total samples   : {len(df)}')
print()

sample_path = train_df.sample(1, random_state=0).iloc[0]['file']
sample      = np.load(sample_path)

print(f'  Sample shape    : {sample.shape}  (expected {(SEQ_LEN, N_KEYPOINTS, N_DIMS)})')
print(f'  Sample dtype    : {sample.dtype}')
print(f'  Value range     : [{sample.min():.3f}, {sample.max():.3f}]')

assert sample.shape == (SEQ_LEN, N_KEYPOINTS, N_DIMS), 'Shape mismatch!'
print('\nAll checks passed.')

=== Pipeline Summary ===
  Classes         : 1245
  Train samples   : 28323
  Val   samples   : 5416
  Test  samples   : 3611
  Total samples   : 37350

  Sample shape    : (60, 75, 2)  (expected (60, 75, 2))
  Sample dtype    : float32
  Value range     : [-2.317, 7.155]

All checks passed.
